In [147]:
# Installation Java
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# Téléchargement Spark
!wget -q https://archive.apache.org/dist/spark/spark-3.4.1/spark-3.4.1-bin-hadoop3.tgz

# Extraction Spark
!tar -xzf spark-3.4.1-bin-hadoop3.tgz

# Installer findspark
!pip install -q findspark



In [148]:
import os
import findspark

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.4.1-bin-hadoop3"

findspark.init()


In [149]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("...") \
    .config("spark.network.timeout", "8000s") \
    .config("spark.executor.heartbeatInterval", "120s") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()


In [201]:
from google.colab import files

uploaded = files.upload()


Saving train_dataset.csv to train_dataset (3).csv


In [202]:
import pandas as pd

df_train=pd.read_csv('train_dataset.csv')


In [203]:
# Étape 1 : Nettoyage des colonnes et filtrage des classes

# Nettoyage des noms de colonnes : suppression des espaces superflus
df_train.columns = df_train.columns.str.strip()

# Filtrage pour ne garder que les classes pertinentes : Benign, DoS, Web Attack (on néglige Infiltration et Heartbleed car échanrillon trop faible et peut donc être vu comme du bruit)
df_train = df_train[df_train['Label'].isin(['Benign', 'DoS', 'Web Attack'])]

# Réinitialiser les index
df_train.reset_index(drop=True, inplace=True)
# 3. Supprimer les colonnes en doublon comme 'Fwd Header Length.1'
df_train = df_train.loc[:, ~df_train.columns.str.contains(r"\.\d+$")]

print(df_train.duplicated().sum())

0


In [204]:
import numpy as np

# Identifier les colonnes pouvant contenir des valeurs infinies ou NaN
# On va d'abord forcer la conversion en float et chercher les valeurs problématiques

# Remplacer les chaînes 'Infinity', 'NaN' (en tant que texte) par np.nan si présentes
df_train.replace(['Infinity', 'NaN', 'inf', '-inf'], np.nan, inplace=True)

# Convertir toutes les colonnes (sauf 'Label') en float si possible
for col in df_train.columns:
    if col != 'Label':
        df_train[col] = pd.to_numeric(df_train[col], errors='coerce')

# Compter les valeurs NaN après nettoyage
nan_counts = df_train.isna().sum()
cols_with_nans = nan_counts[nan_counts > 0]

cols_with_nans

,0


In [205]:
# Supprimer les lignes contenant des NaN (ici uniquement dans 'Flow Bytes/s')
df_train.dropna(inplace=True)

# Vérification après suppression
final_shape = df_train.shape
remaining_nans = df_train.isna().sum().sum()  # Total de NaN restants

final_shape, remaining_nans


((9000, 78), np.int64(0))

In [206]:
from pyspark.sql.functions import col, when, expr
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier, LinearSVC, GBTClassifier, OneVsRest
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


In [207]:
# 3. Conversion en DataFrame Spark
spark_df = spark.createDataFrame(df_train)

In [208]:

from pyspark.sql.functions import monotonically_increasing_id
# 4. Nettoyage des noms de colonnes
spark_df = spark_df.toDF(*[c.strip() for c in spark_df.columns])



# Ajouter une colonne id unique à chaque ligne
spark_df = spark_df.withColumn("id", monotonically_increasing_id())

In [209]:
# 5. Encodage du label déjà présent
indexer = StringIndexer(inputCol="Label", outputCol="label_index", handleInvalid="skip")
spark_df = indexer.fit(spark_df).transform(spark_df)

In [210]:
from pyspark.sql.functions import when

# Total d'exemples
total_count = 5000 + 2500 + 1500

# Fréquence inverse (plus une classe est rare, plus elle a du poids)
spark_df = spark_df.withColumn(
    "classWeightCol",
    when(F.col("label_index") == 0.0, total_count / 5000).
    when(F.col("label_index") == 1.0, total_count / 2500).
    when(F.col("label_index") == 2.0, total_count / 1500)
)


In [211]:
spark_df.select("label_index", "classWeightCol").show(5)


+-----------+--------------+
|label_index|classWeightCol|
+-----------+--------------+
|        0.0|           1.8|
|        0.0|           1.8|
|        0.0|           1.8|
|        0.0|           1.8|
|        0.0|           1.8|
+-----------+--------------+
only showing top 5 rows



In [212]:
# 6. Forcer les colonnes numériques à être bien typées en float
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import isnan, isnull


numeric_cols = [f.name for f in spark_df.schema.fields if f.name not in ["Label", "label_index", "classWeightCol"]]
# 6bis. Supprimer les lignes contenant NaN ou Inf dans les colonnes numériques
for col_name in numeric_cols:
    spark_df = spark_df.filter(~isnan(col(col_name))) \
                       .filter(~isnull(col(col_name))) \
                       .filter(~(col(col_name) == float("inf"))) \
                       .filter(~(col(col_name) == float("-inf")))
for col_name in numeric_cols:
    spark_df = spark_df.withColumn(col_name, col(col_name).cast(DoubleType()))


# 7. Assemblage des features
assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features_assembled")
scaler = StandardScaler(inputCol="features_assembled", outputCol="features")


In [213]:

# 8. Définir les modèles

#Random Forest avec WeightCol
rf = RandomForestClassifier(
    labelCol="label_index",
    featuresCol="features",
    predictionCol="rf_prediction",
    weightCol="classWeightCol",
)
# LinearSVC avec WeightCol
svm_classifier = LinearSVC(
    labelCol="label_index",
    featuresCol="features",
    predictionCol="svm_pred",
)

# OneVsRest avec LinearSvc
svm = OneVsRest(
    classifier=svm_classifier,
    labelCol="label_index",
    featuresCol="features",
    predictionCol="svm_pred"
)

# Classifieur GBT avec weightCoL
gbt_base = GBTClassifier(
    featuresCol="features",
    labelCol="label_index",
    predictionCol="gbt_pred",
    maxIter=20,
    maxDepth=5
)

# OneVsRest avec GBT
gbt = OneVsRest(
    classifier=gbt_base,
    featuresCol="features",
    labelCol="label_index",
    predictionCol="gbt_pred"
)

In [214]:
spark_df.count()

8996

In [215]:
# 9. Pipelines pour chaque modèle
pipeline_rf = Pipeline(stages=[assembler, scaler, rf])
pipeline_svm = Pipeline(stages=[assembler, scaler, svm])
pipeline_gbt = Pipeline(stages=[assembler, scaler, gbt])

In [216]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="label_index", predictionCol="rf_prediction", metricName="f1")

paramGrid_rf = (ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50])
    .addGrid(rf.maxDepth, [5, 10])
    .addGrid(rf.maxBins, [32, 64])
    .build())

cv_rf = CrossValidator(
    estimator=pipeline_rf,
    estimatorParamMaps=paramGrid_rf,
    evaluator=evaluator,
    numFolds=3
)

cv_model_rf = cv_rf.fit(spark_df)



In [226]:
evaluator_svm = MulticlassClassificationEvaluator(labelCol="label_index", predictionCol="svm_pred", metricName="f1")

paramGrid_svm = (ParamGridBuilder()
    .addGrid(svm_classifier.maxIter, [50, 100])
    .addGrid(svm_classifier.regParam, [0.01, 0.1, 1.0])
    .build())

cv_svm = CrossValidator(
    estimator=pipeline_svm,
    estimatorParamMaps=paramGrid_svm,
    evaluator=evaluator_svm,
    numFolds=3
)

cv_model_svm = cv_svm.fit(spark_df)


In [229]:
evaluator_gbt = MulticlassClassificationEvaluator(labelCol="label_index", predictionCol="gbt_pred", metricName="f1")

paramGrid_gbt = (ParamGridBuilder()
    .addGrid(gbt_base.maxIter, [10, 20])
    .addGrid(gbt_base.maxDepth, [3, 5])
    .addGrid(gbt_base.stepSize, [0.05, 0.1])
    .build())

cv_gbt = CrossValidator(
    estimator=pipeline_gbt,
    estimatorParamMaps=paramGrid_gbt,
    evaluator=evaluator_gbt,
    numFolds=3
)

cv_model_gbt = cv_gbt.fit(spark_df)


In [166]:
# 10. Entraîner les modèles
model_rf = pipeline_rf.fit(spark_df)

In [167]:
model_svm = pipeline_svm.fit(spark_df)
model_gbt = pipeline_gbt.fit(spark_df)

In [168]:
from google.colab import files

uploaded2 = files.upload()

Saving test_dataset.csv to test_dataset (3).csv


In [217]:
# Charger le dataset de test
df_test_pandas = pd.read_csv("test_dataset.csv")

# Nettoyage des colonnes dupliquées
df_test_pandas = df_test_pandas.loc[:, ~df_test_pandas.columns.str.contains(r"\.\d+$")]
print(df_test_pandas.duplicated().sum())


0


In [218]:
df_test_pandas[" Label"].value_counts()

,count
Label,
Benign,7400
DoS,2000
Web Attack,600


In [219]:
spark_df_test = spark.createDataFrame(df_test_pandas)
spark_df_test.count()

10000

In [220]:
spark_df_test = spark.createDataFrame(df_test_pandas)

# Nettoyage des noms de colonnes
spark_df_test = spark_df_test.toDF(*[c.strip() for c in spark_df_test.columns])

# Cast des colonnes numériques
numeric_cols = [c for c in spark_df_test.columns if c not in ["Label"]]
for col_name in numeric_cols:
    spark_df_test = spark_df_test.withColumn(col_name, col(col_name).cast(DoubleType()))

spark_df_test = spark_df_test.withColumn("id", monotonically_increasing_id())

In [221]:
spark_df_test = indexer.fit(spark_df_test).transform(spark_df_test)

In [222]:
# Prédictions avec chaque modèle
pred_rf_test = model_rf.transform(spark_df_test).select("id", "label_index", "rf_prediction")
pred_svm_test = model_svm.transform(spark_df_test).select("id", "label_index", "svm_pred")
pred_gbt_test = model_gbt.transform(spark_df_test).select("id", "label_index", "gbt_pred")

In [224]:
predictions_cv_rf_test = cv_model_rf.transform(spark_df_test).select("id", "label_index", "rf_prediction")


In [225]:
evaluator_cv_rf = MulticlassClassificationEvaluator(
    labelCol="label_index", predictionCol="rf_prediction", metricName="f1"
    )
f1_score_cv_rf = evaluator_cv_rf.evaluate(predictions_cv_rf_test)
print(  f"✅ F1-score Random Forest: {f1_score_cv_rf:.4f}")

✅ F1-score Random Forest: 0.9215


In [227]:
predictions_cv_svm_test = cv_model_svm.transform(spark_df_test).select("id", "label_index", "svm_pred")

In [228]:
evaluator_cv_svm = MulticlassClassificationEvaluator(
    labelCol="label_index", predictionCol="svm_pred", metricName="f1"
    )
f1_score_cv_svm = evaluator_cv_svm.evaluate(predictions_cv_svm_test)
print(  f"✅ F1-score SVM: {f1_score_cv_svm:.4f}")

✅ F1-score SVM: 0.9047


In [230]:
predictions_cv_gbt_test = cv_model_gbt.transform(spark_df_test).select("id", "label_index", "gbt_pred")

In [231]:
evaluator_cv_gbt = MulticlassClassificationEvaluator(
    labelCol="label_index", predictionCol="gbt_pred", metricName="f1"
    )
f1_score_cv_gbt = evaluator_cv_gbt.evaluate(predictions_cv_gbt_test)
print(  f"✅ F1-score GBT: {f1_score_cv_gbt:.4f}")

✅ F1-score GBT: 0.6444


In [232]:
predictions_test = predictions_cv_rf_test.select("id", "label_index", "rf_prediction") \
    .join(predictions_cv_svm_test.select("id", "svm_pred"), on="id") \
    .join(predictions_cv_gbt_test.select("id", "gbt_pred"), on="id")

In [233]:
from pyspark.sql.functions import when, col

# Créer des colonnes de vote pour chaque classe
predictions_test = predictions_test.withColumn("vote_0",
    (col("rf_prediction") == 0).cast("int") +
    (col("svm_pred") == 0).cast("int") +
    (col("gbt_pred") == 0).cast("int")
)

predictions_test = predictions_test.withColumn("vote_1",
    (col("rf_prediction") == 1).cast("int") +
    (col("svm_pred") == 1).cast("int") +
    (col("gbt_pred") == 1).cast("int")
)

predictions_test = predictions_test.withColumn("vote_2",
    (col("rf_prediction") == 2).cast("int") +
    (col("svm_pred") == 2).cast("int") +
    (col("gbt_pred") == 2).cast("int")
)

# Choisir la classe avec le plus de votes
predictions_test = predictions_test.withColumn(
    "final_prediction",
    when((col("vote_0") >= col("vote_1")) & (col("vote_0") >= col("vote_2")), 0.0)
    .when((col("vote_1") >= col("vote_0")) & (col("vote_1") >= col("vote_2")), 1.0)
    .when((col("vote_2") >= col("vote_0")) & (col("vote_2") >= col("vote_1")), 2.0)
)


In [234]:
predictions_test.select("label_index","rf_prediction" , "svm_pred", "gbt_pred", "final_prediction").show(100)

+-----------+-------------+--------+--------+----------------+
|label_index|rf_prediction|svm_pred|gbt_pred|final_prediction|
+-----------+-------------+--------+--------+----------------+
|        1.0|          0.0|     0.0|     0.0|             0.0|
|        0.0|          0.0|     0.0|     0.0|             0.0|
|        0.0|          0.0|     0.0|     0.0|             0.0|
|        0.0|          0.0|     0.0|     0.0|             0.0|
|        1.0|          1.0|     1.0|     1.0|             1.0|
|        0.0|          0.0|     0.0|     0.0|             0.0|
|        0.0|          0.0|     0.0|     0.0|             0.0|
|        0.0|          0.0|     0.0|     0.0|             0.0|
|        0.0|          0.0|     0.0|     0.0|             0.0|
|        0.0|          0.0|     0.0|     0.0|             0.0|
|        1.0|          0.0|     0.0|     0.0|             0.0|
|        0.0|          0.0|     0.0|     0.0|             0.0|
|        0.0|          0.0|     0.0|     0.0|          

In [235]:
predictions_disagree = predictions_test.filter(
    (col("vote_0") == 1) &
    (col("vote_1") == 1) &
    (col("vote_2") == 1)
)
print(f"🎯 Nombre de cas où chaque modèle vote pour une classe différente : {predictions_disagree.count()}")


🎯 Nombre de cas où chaque modèle vote pour une classe différente : 107


In [236]:
predictions_disagree.filter(col("label_index") == col("final_prediction")).count()

102

In [237]:
predictions_test.count()

10000

In [197]:
evaluator1 = MulticlassClassificationEvaluator(
    labelCol="label_index", predictionCol="final_prediction", metricName="f1"
    )
f1_score1 = evaluator1.evaluate(predictions_test)

evaluator2 = MulticlassClassificationEvaluator(
    labelCol="label_index", predictionCol="rf_prediction", metricName="f1"
    )
f1_score2 = evaluator2.evaluate(pred_rf_test)

evaluator3 = MulticlassClassificationEvaluator(
    labelCol="label_index", predictionCol="svm_pred", metricName="f1"
    )
f1_score3 = evaluator3.evaluate(pred_svm_test)

evaluator4 = MulticlassClassificationEvaluator(
    labelCol="label_index", predictionCol="gbt_pred", metricName="f1"
    )
f1_score4 = evaluator4.evaluate(pred_gbt_test)

print(f"✅ F1-score with majority voting: {f1_score1:.4f}")
print(f"✅ F1-score Random Forest: {f1_score2:.4f}")
print(f"✅ F1-score SVM: {f1_score3:.4f}")
print(f"✅ F1-score GBT: {f1_score4:.4f}")

✅ F1-score with majority voting: 0.7774
✅ F1-score Random Forest: 0.7674
✅ F1-score SVM: 0.9035
✅ F1-score GBT: 0.6382


In [239]:
correct_preds_test = predictions_test.filter(col("final_prediction") == col("label_index")).count()
total_preds = predictions_test.count()
accuracy1 = correct_preds_test / total_preds * 100

correct_preds_rf = predictions_cv_rf_test.filter(col("rf_prediction") == col("label_index")).count()
total_preds_rf = predictions_cv_rf_test.count()
accuracy2 = correct_preds_rf / total_preds_rf * 100

correct_preds_svm = predictions_cv_svm_test.filter(col("svm_pred") == col("label_index")).count()
total_preds_svm = predictions_cv_svm_test.count()
accuracy3 = correct_preds_svm / total_preds_svm * 100

correct_preds_gbt = predictions_cv_gbt_test.filter(col("gbt_pred") == col("label_index")).count()
total_preds_gbt = predictions_cv_gbt_test.count()
accuracy4 = correct_preds_gbt / total_preds_gbt * 100




print(f"✅ Accuracy with majority voting: {accuracy1:.2f}%")
print(f"✅ Accuracy Random Forest: {accuracy2:.2f}%")
print(f"✅ Accuracy SVM: {accuracy3:.2f}%")
print(f"✅ Accuracy GBT: {accuracy4:.2f}%")



✅ Accuracy with majority voting: 91.33%
✅ Accuracy Random Forest: 92.66%
✅ Accuracy SVM: 90.90%
✅ Accuracy GBT: 60.54%


In [242]:
from pyspark.sql import functions as F
import seaborn as sns
import matplotlib.pyplot as plt

confusion_df_vote = predictions_test.groupBy("label_index", "final_prediction").count().orderBy("label_index", "final_prediction")
confusion_df_vote.show()

confusion_df_rf = predictions_cv_rf_test.groupBy("label_index", "rf_prediction").count().orderBy("label_index", "rf_prediction")
confusion_df_rf.show()

confusion_df_svm = predictions_cv_svm_test.groupBy("label_index", "svm_pred").count().orderBy("label_index", "svm_pred")
confusion_df_svm.show()

confusion_df_gbt = predictions_cv_gbt_test.groupBy("label_index", "gbt_pred").count().orderBy("label_index", "gbt_pred")
confusion_df_gbt.show()

+-----------+----------------+-----+
|label_index|final_prediction|count|
+-----------+----------------+-----+
|        0.0|             0.0| 7113|
|        0.0|             1.0|  108|
|        0.0|             2.0|  179|
|        1.0|             0.0|  194|
|        1.0|             1.0| 1802|
|        1.0|             2.0|    4|
|        2.0|             0.0|  381|
|        2.0|             1.0|    1|
|        2.0|             2.0|  218|
+-----------+----------------+-----+

+-----------+-------------+-----+
|label_index|rf_prediction|count|
+-----------+-------------+-----+
|        0.0|          0.0| 7183|
|        0.0|          1.0|   56|
|        0.0|          2.0|  161|
|        1.0|          0.0|  145|
|        1.0|          1.0| 1853|
|        1.0|          2.0|    2|
|        2.0|          0.0|  369|
|        2.0|          1.0|    1|
|        2.0|          2.0|  230|
+-----------+-------------+-----+

+-----------+--------+-----+
|label_index|svm_pred|count|
+-----------+----